# 🏥 NoShowAI — Medical Appointment No-Show Prediction

**An AI-powered system that predicts whether a patient will miss their clinical appointment,**
using historical scheduling data, machine-learning models, and an interactive Streamlit dashboard.

---

## 👥 Team

| Member | Role |
|--------|------|
| **Harsh** | Data Preprocessing & Feature Engineering |
| **Manmath** | ML Modeling & Evaluation |
| **Yash** | UI — Streamlit Dashboard |
| **Ansh** | Analysis & Feature Importance / Visualisation |

---

## 📌 Problem Statement

Missed clinical appointments (no-shows) are a major operational problem for healthcare providers:

- **Resource wastage** — staff and equipment sit idle
- **Increased waiting times** — other patients can't fill the slot in time
- **Revenue loss** — unpaid/wasted consultation slots
- **Disrupted care continuity** — patients miss needed follow-ups

This project trains a supervised ML classifier to **predict no-show risk per appointment**
so that clinics can send targeted reminders, reschedule proactively, or overbook safely.

---

## 📂 Dataset

**Source:** Kaggle — *Medical Appointment No Shows* (`KaggleV2-May-2016.csv`, Brazil 2016)  
**Raw size:** ~110 000 appointments

| Original Column | Renamed To | Description |
|-----------------|-----------|-------------|
| `No-show` | `NoShow` | **Target** — Yes / No → 1 / 0 |
| `Hipertension` | `Hypertension` | Chronic hypertension flag |
| `Handcap` | `Handicap` | Disability level (0–4) |
| `PatientId` | *(dropped)* | Not useful for ML |
| `AppointmentID` | *(dropped)* | Not useful for ML |
| `ScheduledDay` | — | When the appointment was booked |
| `AppointmentDay` | — | The actual appointment date |
| `Age` | — | Patient age |
| `Gender` | `Gender_*` | one-hot encoded |
| `Neighbourhood` | *(one-hot, then dropped)* | Location |
| `Scholarship` | — | Bolsa Família welfare (0/1) |
| `Diabetes` | — | Diabetes flag (0/1) |
| `Alcoholism` | — | Alcoholism flag (0/1) |
| `SMS_received` | — | Reminder SMS received (0/1) |


---
# 🔧 PART 1 — Data Preprocessing *(Harsh)*

All preprocessing was originally performed in **Google Colab** and the cleaned CSV was saved
as `data/noshow_cleaned.csv`. The full pipeline is reproduced here for transparency.

**Steps:**
1. Load raw Kaggle CSV
2. Rename problematic column names
3. Check & handle missing values
4. Convert date columns
5. Feature engineering (LeadTime, Weekday, IsWeekend)
6. Remove useless ID columns
7. Convert target column Yes/No → 0/1
8. Encode categorical columns (Gender, Neighbourhood)
9. Remove invalid data (negative lead times)
10. Save cleaned CSV


In [ ]:
# Install dependencies (run once)
%pip install pandas numpy matplotlib seaborn scikit-learn joblib --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (9, 5)
print('Libraries loaded ✔')

## Step 1 — Load Raw CSV


In [ ]:
# In Colab: upload KaggleV2-May-2016.csv and move it to data/
# Here we load directly (adjust path if needed)
try:
    df = pd.read_csv('data/KaggleV2-May-2016.csv')
    print('Loaded raw dataset ✔')
except FileNotFoundError:
    # Fallback: use the already-cleaned file
    df = pd.read_csv('data/cleaned_data.csv')
    print('Raw CSV not found — loaded cleaned_data.csv instead')

print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

## Step 2 — Rename Problematic Columns

The raw Kaggle CSV has typos/dashes that break Python attribute access and ML pipelines.
We fix them immediately:

| Before | After |
|--------|-------|
| `No-show` | `NoShow` |
| `Hipertension` | `Hypertension` |
| `Handcap` | `Handicap` |


In [ ]:
df.rename(columns={
    'No-show':     'NoShow',
    'Hipertension':'Hypertension',
    'Handcap':     'Handicap'
}, inplace=True)
print('Columns renamed ✔')
print(df.columns.tolist())

## Step 3 — Check Missing Values

The Kaggle dataset is generally clean, but we verify and drop any rows with nulls.


In [ ]:
print('Missing values per column:')
print(df.isnull().sum())

if df.isnull().any().any():
    before = len(df)
    df.dropna(inplace=True)
    print(f'Dropped {before - len(df)} rows with missing values')
else:
    print('No missing values found ✔')

## Step 4 — Convert Date Columns

`ScheduledDay` and `AppointmentDay` are strings — parse them as datetime
so we can compute time-based features in the next step.


In [ ]:
df['ScheduledDay']   = pd.to_datetime(df['ScheduledDay'])
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])
print('Date columns converted ✔')
df[['ScheduledDay','AppointmentDay']].dtypes

## Step 5 — Feature Engineering ⭐ *(Most Important Step)*

Three new features created by Harsh that dramatically boost model performance:

| New Feature | Formula | Why it matters |
|-------------|---------|----------------|
| `LeadTime` | `(AppointmentDay − ScheduledDay).dt.days` | Longer wait → higher no-show risk |
| `AppointmentWeekday` | `.dt.dayofweek` (0=Mon … 6=Sun) | Weekdays differ in no-show rate |
| `IsWeekend` | 1 if weekday ≥ 5 | Weekend appointments are rarer & riskier |


In [ ]:
# Lead time = waiting days between booking and appointment
df['LeadTime'] = (df['AppointmentDay'] - df['ScheduledDay']).dt.days

# Day of week the appointment falls on (0=Monday)
df['AppointmentWeekday'] = df['AppointmentDay'].dt.dayofweek

# Weekend flag
df['IsWeekend'] = df['AppointmentWeekday'].apply(lambda x: 1 if x >= 5 else 0)

print('New features:')
df[['LeadTime','AppointmentWeekday','IsWeekend']].describe()

## Step 6 — Remove Useless Columns

`PatientId` and `AppointmentID` are unique identifiers — they carry no predictive signal
and would cause data leakage if kept.


In [ ]:
cols_to_drop = [c for c in ['PatientId','AppointmentID'] if c in df.columns]
df.drop(columns=cols_to_drop, inplace=True)
print(f'Dropped: {cols_to_drop} ✔')

## Step 7 — Convert Target Column (Yes/No → 1/0)

ML models need numeric labels.  
**`NoShow = 1`** means the patient did **not** attend the appointment.


In [ ]:
if df['NoShow'].dtype == object:
    df['NoShow'] = df['NoShow'].map({'Yes': 1, 'No': 0})
    print('Target encoded: Yes→1, No→0 ✔')
else:
    print('Target already numeric ✔')
df['NoShow'].value_counts()

## Step 8 — Encode Categorical Columns

We use **`pd.get_dummies`** to one-hot encode `Gender` and `Neighbourhood`.
Then we drop `Neighbourhood` columns (too many categories, adds noise) and keep only `Gender_M`.


In [ ]:
cols_to_encode = [c for c in ['Gender','Neighbourhood'] if c in df.columns]
if cols_to_encode:
    df = pd.get_dummies(df, columns=cols_to_encode, drop_first=True)
    print(f'One-hot encoded: {cols_to_encode} ✔')

# Drop Neighbourhood dummy columns — too many, add noise
neighbourhood_cols = [c for c in df.columns if c.startswith('Neighbourhood_')]
if neighbourhood_cols:
    df.drop(columns=neighbourhood_cols, inplace=True)
    print(f'Dropped {len(neighbourhood_cols)} Neighbourhood dummies')

# Rename Gender_M if present (drop_first=True keeps the Male column)
gender_col = [c for c in df.columns if c.startswith('Gender_')]
print(f'Gender columns kept: {gender_col}')
df.head(3)

## Step 9 — Remove Invalid Data

Negative `LeadTime` means the appointment was scheduled *after* it was supposed to happen —
a data entry error. These rows must be removed.


In [ ]:
before = len(df)
df = df[df['LeadTime'] >= 0]
print(f'Removed {before - len(df)} rows with negative LeadTime')
print(f'Clean dataset: {df.shape}')

## Step 10 — Final Check


In [ ]:
print('=== Final Dataset Info ===')
df.info()
print()
print('Missing values:', df.isnull().sum().sum())
print('All numeric:', all(df.dtypes != object))
print('Target column exists:', 'NoShow' in df.columns)
df.head()

## Step 11 — Save Cleaned Dataset


In [ ]:
import os
os.makedirs('data', exist_ok=True)
df.to_csv('data/noshow_cleaned.csv', index=False)
print('Saved → data/noshow_cleaned.csv ✔')

### ✅ Preprocessing Summary

| Step | Action | Result |
|------|--------|--------|
| Column rename | `No-show→NoShow`, `Hipertension→Hypertension`, `Handcap→Handicap` | Clean names |
| Missing values | Checked, drop rows with nulls | 0 missing |
| Date parsing | `ScheduledDay` & `AppointmentDay` → datetime | Time arithmetic enabled |
| **Feature engineering** | `LeadTime`, `AppointmentWeekday`, `IsWeekend` | **Key predictors created** |
| Drop IDs | `PatientId`, `AppointmentID` removed | No leakage |
| Target encoding | `Yes→1`, `No→0` | Numeric labels |
| One-hot encoding | `Gender` kept, `Neighbourhood` dropped | Manageable feature space |
| Invalid data | `LeadTime < 0` rows removed | Clean data |

**Final dataset: `data/noshow_cleaned.csv`** — fully numeric, zero nulls, ready for ML.


---
# 📊 PART 2 — Exploratory Data Analysis & Feature Importance *(Ansh)*

**Responsibilities:**
- Explain *why* no-shows happen through visualisation
- Identify the most powerful predictors
- Surface actionable insights for clinics


In [ ]:
# Work with the cleaned dataset
df = pd.read_csv('data/noshow_cleaned.csv')
print(f'Loaded cleaned dataset: {df.shape}')

## 2.1 Class Distribution


In [ ]:
counts = df['NoShow'].value_counts()
labels = ['Showed Up (0)', 'No-Show (1)']

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(x='NoShow', data=df, palette=['#4CAF50','#F44336'], ax=axes[0])
axes[0].set_title('Appointment Outcomes — Count')
axes[0].set_xticklabels(labels)

axes[1].pie(counts, labels=labels, autopct='%1.1f%%',
            colors=['#4CAF50','#F44336'], startangle=140)
axes[1].set_title('Appointment Outcomes — Share')

plt.suptitle('Class Imbalance: ~71.5% Show vs ~28.5% No-Show', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print(counts)

**Key insight:** The dataset is imbalanced — only ~28.5 % are no-shows.
We must use **F1-score / recall** (not plain accuracy) to evaluate models.


## 2.2 Age vs No-Show


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(x='NoShow', y='Age', data=df, palette=['#4CAF50','#F44336'], ax=axes[0])
axes[0].set_xticklabels(['Showed Up', 'No-Show'])
axes[0].set_title('Age Distribution by Outcome')

for val, color, label in [(0,'#4CAF50','Showed Up'), (1,'#F44336','No-Show')]:
    axes[1].hist(df[df['NoShow']==val]['Age'], bins=30,
                 alpha=0.6, color=color, label=label, density=True)
axes[1].legend()
axes[1].set_title('Age Density by Outcome')
axes[1].set_xlabel('Age')

plt.tight_layout()
plt.show()

**Key insight:** Younger patients (20–40) have a **higher no-show rate**.
Older patients with chronic conditions tend to show up more reliably.


## 2.3 Lead Time vs No-Show *(Top Predictor)*


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(x='NoShow', y='LeadTime', data=df, palette=['#4CAF50','#F44336'], ax=axes[0])
axes[0].set_xticklabels(['Showed Up', 'No-Show'])
axes[0].set_title('Lead Time vs Outcome')
axes[0].set_ylabel('Days between booking & appointment')

df['LeadBucket'] = pd.cut(df['LeadTime'], bins=[-1,0,7,30,60,500],
                           labels=['Same day','1-7 d','8-30 d','31-60 d','60+ d'])
rate = df.groupby('LeadBucket', observed=True)['NoShow'].mean() * 100
rate.plot(kind='bar', color='#2196F3', ax=axes[1], rot=30)
axes[1].set_title('No-Show Rate by Lead Time Bucket')
axes[1].set_ylabel('No-Show Rate (%)')

plt.tight_layout()
plt.show()
df.drop(columns=['LeadBucket'], inplace=True, errors='ignore')

**Key insight:** Lead time is the **strongest predictor**.
Same-day appointments almost never result in no-shows.
Appointments booked 60+ days ahead have the highest no-show rate.


## 2.4 SMS Reminder Impact


In [ ]:
pivot = df.groupby('SMS_received')['NoShow'].value_counts(normalize=True).unstack() * 100
pivot.index = ['No SMS','SMS sent']
pivot.columns = ['Showed Up (%)','No-Show (%)']

pivot.plot(kind='bar', color=['#4CAF50','#F44336'], rot=0, figsize=(7, 4))
plt.title('SMS Reminder vs No-Show Rate')
plt.ylabel('Percentage')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()
print(pivot.round(1))

**Key insight (counter-intuitive):** Patients who received an SMS had a slightly *higher* no-show rate.
Why? SMS reminders were sent to high-risk (long lead-time) appointments.
This is a **confounding variable** — lead time is the real driver.


## 2.5 Appointment Weekday vs No-Show


In [ ]:
day_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
rate_by_day = df.groupby('AppointmentWeekday')['NoShow'].mean() * 100
rate_by_day.index = [day_names[i] for i in rate_by_day.index]

rate_by_day.plot(kind='bar', color='#9C27B0', rot=0, figsize=(8, 4))
plt.title('No-Show Rate by Appointment Day of Week')
plt.ylabel('No-Show Rate (%)')
plt.tight_layout()
plt.show()

## 2.6 Health Conditions & Socioeconomic Factors


In [ ]:
conditions = ['Hypertension','Diabetes','Alcoholism','Scholarship']
conditions = [c for c in conditions if c in df.columns]

fig, axes = plt.subplots(1, len(conditions), figsize=(14, 4))
if len(conditions) == 1:
    axes = [axes]

for ax, col in zip(axes, conditions):
    rate = df.groupby(col)['NoShow'].mean() * 100
    rate.plot(kind='bar', ax=ax, color=['#4CAF50','#F44336'], rot=0)
    ax.set_title(col)
    ax.set_ylabel('No-Show Rate (%)' if col == conditions[0] else '')
    ax.set_xticklabels(['No','Yes'])

plt.suptitle('No-Show Rate by Condition / Socioeconomic Factor', y=1.02)
plt.tight_layout()
plt.show()

## 2.7 Feature Importance (Random Forest)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

drop_cols = ['NoShow','ScheduledDay','AppointmentDay']
FEATURES = [c for c in df.columns if c not in drop_cols and df[c].dtype != object]

X = df[FEATURES]
y = df['NoShow']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

rf_fi = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_fi.fit(X_train, y_train)

importances = pd.Series(rf_fi.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, max(4, len(FEATURES)*0.4)))
importances.plot(kind='barh', color='#FF5722', ax=ax)
ax.set_title('Feature Importance — Random Forest')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Top 5 features:')
print(importances.sort_values(ascending=False).head(5))

### 📌 Feature Importance Summary

| Rank | Feature | Insight |
|------|---------|--------|
| 1 | **LeadTime** | Engineered by Harsh — strongest predictor |
| 2 | **Age** | Younger patients miss more |
| 3 | **AppointmentWeekday** | Day of week matters |
| 4 | **SMS_received** | Correlated with high lead time |
| 5 | **Scholarship** | Socioeconomic proxy |


---
# 🤖 PART 3 — Machine Learning Modeling *(Manmath)*

**Responsibilities:**
- Train Logistic Regression, Decision Tree, Random Forest
- Handle class imbalance (`class_weight='balanced'` + optional SMOTE)
- Evaluate with Accuracy, F1-Score, classification report
- Select and save the best model


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, f1_score,
                              classification_report, ConfusionMatrixDisplay)
import joblib, os

drop_cols = ['NoShow','ScheduledDay','AppointmentDay']
FEATURES = [c for c in df.columns if c not in drop_cols and df[c].dtype != object]

X = df[FEATURES]
y = df['NoShow']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print(f'Train: {len(X_train)} rows  |  Test: {len(X_test)} rows')

## 3.1 Handling Class Imbalance

No-shows ≈ 28.5 % of data. Without correction a naive model just predicts *show-up* always.
We use `class_weight='balanced'` (built-in to sklearn) and optionally SMOTE oversampling.


In [ ]:
try:
    from imblearn.over_sampling import SMOTE
    smote = SMOTE(random_state=42)
    X_res, y_res   = smote.fit_resample(X_train, y_train)
    X_res_sc, _    = smote.fit_resample(X_train_sc, y_train)
    print(f'SMOTE applied — balanced train size: {len(X_res)}')
    use_smote = True
except ImportError:
    print('imblearn not installed — using class_weight="balanced" only')
    X_res, y_res = X_train, y_train
    X_res_sc     = X_train_sc
    use_smote    = False

## 3.2 Train All Three Models


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Decision Tree':       DecisionTreeClassifier(max_depth=10, random_state=42, class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
}

results  = []
trained  = {}

for name, model in models.items():
    tr_X = X_res_sc if name == 'Logistic Regression' else X_res
    te_X = X_test_sc if name == 'Logistic Regression' else X_test
    tr_y = y_res

    model.fit(tr_X, tr_y)
    y_pred = model.predict(te_X)

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, pos_label=1)
    results.append({'Model': name, 'Accuracy': round(acc,4), 'F1 (No-Show)': round(f1,4)})
    trained[name] = model
    print(f'{name:22s}  Acc={acc:.4f}  F1={f1:.4f}')

results_df = pd.DataFrame(results).sort_values('F1 (No-Show)', ascending=False)
print()
print(results_df.to_string(index=False))

## 3.3 Model Comparison Chart


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

results_df.plot(kind='bar', x='Model', y='Accuracy',
                color='#2196F3', ax=axes[0], rot=20, legend=False)
axes[0].set_title('Model Accuracy')
axes[0].set_ylim(0, 1)

results_df.plot(kind='bar', x='Model', y='F1 (No-Show)',
                color='#F44336', ax=axes[1], rot=20, legend=False)
axes[1].set_title('F1-Score (No-Show class)')
axes[1].set_ylim(0, 1)

plt.suptitle('Model Comparison', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 3.4 Detailed Classification Reports


In [ ]:
for name, model in trained.items():
    X_te = X_test_sc if name == 'Logistic Regression' else X_test
    y_pred = model.predict(X_te)
    print('='*52)
    print(f'  {name}')
    print('='*52)
    print(classification_report(y_test, y_pred, target_names=['Show (0)','No-Show (1)']))

## 3.5 Confusion Matrix — Best Model


In [ ]:
best_name  = results_df.iloc[0]['Model']
best_model = trained[best_name]
X_te_best  = X_test_sc if best_name == 'Logistic Regression' else X_test
y_pred_best = best_model.predict(X_te_best)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best,
    display_labels=['Show','No-Show'],
    cmap='Blues', ax=ax)
ax.set_title(f'Confusion Matrix — {best_name}')
plt.tight_layout()
plt.show()
print(f'Best model: {best_name}')

## 3.6 Save Best Model & Scaler


In [ ]:
os.makedirs('model', exist_ok=True)
joblib.dump(scaler, 'model/scaler.pkl')
joblib.dump(best_model, 'model/best_model.pkl')
for name, m in trained.items():
    fname = 'model/' + name.lower().replace(' ','_') + '.pkl'
    joblib.dump(m, fname)
print('All models saved to model/ ✔')

### 🏆 Modeling Summary

| Model | Strength | Weakness |
|-------|----------|----------|
| Logistic Regression | Fast, interpretable, baseline | Lower F1 on no-show class |
| Decision Tree | Explainable, no scaling needed | Can overfit |
| **Random Forest** | Best F1, robust to noise | Slower, less interpretable |

**Chosen model: Random Forest** — highest F1 on the minority no-show class.


---
# 💻 PART 4 — Streamlit Dashboard *(Yash)*

**Responsibilities:**
- Upload CSV of appointments via sidebar
- Show predictions with `NoShow_Probability` and `Risk_Level` labels
- Display summary charts

## 4.1 UI Architecture

```
ui/app.py
  ├─ Sidebar: CSV upload widget
  ├─ Main panel
  │    ├─ Raw data preview
  │    ├─ Prediction table (sorted by risk)
  │    └─ Charts: risk distribution, probability histogram
  └─ Loads:  model/best_model.pkl  +  model/scaler.pkl
```

## 4.2 Streamlit App Code

```python
# ui/app.py
import streamlit as st
import pandas as pd
import joblib

FEATURES = ['Age','Scholarship','Hypertension','Diabetes',
            'Alcoholism','Handicap','SMS_received',
            'LeadTime','AppointmentWeekday','IsWeekend']

st.title('🏥 NoShowAI — Appointment Risk Dashboard')

uploaded = st.sidebar.file_uploader('Upload appointments CSV', type='csv')
if uploaded:
    df = pd.read_csv(uploaded)
    model = joblib.load('model/best_model.pkl')

    proba = model.predict_proba(df[FEATURES])[:, 1]
    df['NoShow_Probability'] = proba
    df['Risk_Level'] = pd.cut(
        proba, bins=[0, 0.3, 0.6, 1.0],
        labels=['Low 🟢', 'Medium 🟡', 'High 🔴'])

    st.subheader('Predictions')
    st.dataframe(df.sort_values('NoShow_Probability', ascending=False))
    st.bar_chart(df['Risk_Level'].value_counts())
```

## 4.3 Run the Dashboard

```bash
pip install streamlit
streamlit run ui/app.py
```

## 4.4 Prediction Simulation


In [ ]:
# Simulate what the UI shows — top 10 test predictions
sample = X_test.copy().reset_index(drop=True).head(10)
proba  = best_model.predict_proba(sample)[:, 1]

sample['NoShow_Probability'] = proba.round(3)
sample['Risk_Level'] = pd.cut(proba, bins=[0,0.3,0.6,1.0],
                               labels=['Low','Medium','High'])
print('Sample prediction output (as shown in the Streamlit UI):')
sample[['Age','LeadTime','SMS_received','NoShow_Probability','Risk_Level']]

In [ ]:
# Risk distribution chart (mirrors the UI bar chart)
all_proba = best_model.predict_proba(X_test)[:, 1]
risk = pd.cut(all_proba, bins=[0,0.3,0.6,1.0], labels=['Low 🟢','Medium 🟡','High 🔴'])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
risk.value_counts()[['Low 🟢','Medium 🟡','High 🔴']].plot(
    kind='bar', color=['green','gold','red'], rot=0, ax=axes[0])
axes[0].set_title('Risk Level Distribution — Test Set')
axes[0].set_ylabel('Number of Appointments')

axes[1].hist(all_proba, bins=30, color='#2196F3', edgecolor='white')
axes[1].set_title('No-Show Probability Distribution')
axes[1].set_xlabel('Predicted Probability')

plt.tight_layout()
plt.show()

---
# 🎯 PART 5 — Summary & Key Takeaways

## 5.1 End-to-End Pipeline

```
KaggleV2-May-2016.csv (raw)
  ↓ [Harsh]   Rename → Missing check → Date parse → Feature engineering
  ↓           Drop IDs → Encode target → One-hot → Remove invalid rows
data/noshow_cleaned.csv
  ↓ [Ansh]    EDA: class balance, age, lead time, SMS, weekday, feature importance
  ↓ [Manmath] Train LR / DT / RF → Evaluate → Save best model
model/best_model.pkl
  ↓ [Yash]    Streamlit dashboard — upload CSV → predictions → risk labels → charts
```

## 5.2 Results At a Glance

| Metric | Value |
|--------|-------|
| Raw dataset | ~110 000 appointments |
| Cleaned dataset | ~71 959 rows × 11+ columns |
| No-show rate | ~28.5 % |
| Best model | Random Forest |
| #1 predictor | `LeadTime` (engineered by Harsh) |

## 5.3 Actionable Insights for Clinics

1. **Long lead-time → high risk** — target extra reminders for appointments booked 2+ weeks ahead
2. **Young patients (20–40)** are the highest-risk group
3. **Same-day bookings** almost never result in a no-show — no need to over-book these
4. SMS reminders alone don't reduce no-shows — **timing and personalisation** matter more

## 5.4 Future Work

- Add patient-level history (repeat no-show count)
- Hyperparameter tuning with `GridSearchCV`
- Deploy Streamlit app to Streamlit Cloud
- SHAP values for per-patient explainability
